# Week 2, day 5 (morning) — Worksheet 07 SOLUTIONS: control flow in a pipeline   (L05)

Every cell below was executed on the same Python the lab ships (3.13), and the
quoted output is what it actually printed.

Read Q5, Q8 and Q11 together. Each one produces output that would pass a code
review, and two of them are lying by omission.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 07 — Control flow in a pipeline. Run this once.

# 23 record ids waiting to be shipped, for the batching questions.
record_ids = list(range(101, 124))
BATCH_SIZE = 5

# A raw feed, exactly as it arrived: amounts are TEXT, and four rows are bad.
feed = [
    {"id": 101, "ts": 5,  "user": "u1", "amount": "12.50"},
    {"id": 102, "ts": 9,  "user": "u2", "amount": "8.00"},
    {"id": 103, "ts": 14, "user": "u1", "amount": ""},        # blank
    {"id": 104, "ts": 20, "user": "u9", "amount": "31.00"},   # user not on file
    {"id": 105, "ts": 26, "user": "u2", "amount": "-4.00"},   # negative
    {"id": 106, "ts": 31, "user": "u3", "amount": "17.25"},
    {"id": 107, "ts": 38, "user": "u1", "amount": "n/a"},     # not a number
    {"id": 108, "ts": 44, "user": "u3", "amount": "9.99"},
]

# The dimension table the feed is joined to.
customers = {"u1": "north", "u2": "south", "u3": "north"}

# What the delivery endpoint returned, attempt by attempt.
attempt_log = ["timeout", "timeout", "ok"]
flaky_log = ["timeout", "timeout", "timeout", "timeout", "timeout"]

MAX_ATTEMPTS = 5
watermark = 0     # the highest ts already processed on a previous run

print(len(record_ids), "ids,", len(feed), "feed rows,",
      len(customers), "customers on file")

PART A — Batching

### Question 1

Batching 23 ids by 5. -> four batches of `5` and a final batch of `3` (`[121, 122, 123]`), then `5 batches from 23 ids`.

`range(0, len(x), BATCH_SIZE)` gives the **start position** of each batch —
0, 5, 10, 15, 20 — and the slice takes it from there.

The last batch is the one to look at. `record_ids[20:25]` asks for five
items from a list that only has three left, and it does not raise: a slice
that runs off the end stops quietly at the end. That is why this pattern
needs no special case for the remainder, and it is one of the few places
where Python's silence is doing you a favour.

(Compare `record_ids[25]` — a single index past the end raises
`IndexError` immediately. Slices forgive, indexes do not.)

In [ ]:
batches = 0
for start in range(0, len(record_ids), BATCH_SIZE):
    batch = record_ids[start:start + BATCH_SIZE]
    batches = batches + 1
    print("batch", batches, "->", batch, "size", len(batch))

print(batches, "batches from", len(record_ids), "ids")

### Question 2

Checking the arithmetic. -> `23 == 23 -> True`. Then `1 -> 23 batches`, `5 -> 5`, `23 -> 1`, and **`50 -> 1 batches`**.

The sizes add back to 23, which is the check worth writing every time you
chunk something. Batching is where rows go missing — an off-by-one in the
slice loses a row per batch and every batch still looks plausible.

A batch size of 50 gives **one** batch, not zero: `range(0, 23, 50)` yields
just `0`, and the slice `[0:50]` hands back all 23. So an oversized batch
degrades to "everything in one go", which is the sensible outcome.

Size 23 also gives one batch, exactly. If it had given two — one full and
one empty — you would be shipping an empty payload on every evenly divided
run.

In [ ]:
shipped = 0
for start in range(0, len(record_ids), BATCH_SIZE):
    shipped = shipped + len(record_ids[start:start + BATCH_SIZE])

print(shipped, "==", len(record_ids), "->", shipped == len(record_ids))

for size in [1, 5, 23, 50]:
    count = len(range(0, len(record_ids), size))
    print("batch size", size, "->", count, "batches")

PART B — Retrying

### Question 3

A retry that succeeds. -> two `timeout`s, then `ok`, then `delivered: True after 3 attempts`.

Two things bound this loop and both are needed: `attempts < MAX_ATTEMPTS`
stops it running forever, and `break` stops it retrying something that has
already worked.

`delivered` is set **inside** the loop and read **after** it. That is
worksheet 04 Q7's flag, and Q4 is about to show why it is not optional.

Real retries wait between attempts, and wait longer each time — that is
called backoff. The loop shape does not change; a `time.sleep()` goes at
the top of the body.

In [ ]:
attempts = 0
delivered = False

while attempts < MAX_ATTEMPTS:
    result = attempt_log[attempts]
    attempts = attempts + 1
    print("attempt", attempts, "->", result)
    if result == "ok":
        delivered = True
        break

print("delivered:", delivered, "after", attempts, "attempts")

### Question 4

A retry that runs out. -> five `timeout`s, then `delivered: False after 5 attempts`.

The loop ended cleanly. It made every attempt it was allowed, nothing
raised, and the delivery **did not happen**.

Put the two final lines side by side: `delivered: True after 3 attempts`
and `delivered: False after 5 attempts`. One word. Without the flag, both
runs end the same way — the loop finishes and the next line of code runs —
and the pipeline carries on as though the data had arrived.

This is the most expensive bug in this whole worksheet, because it produces
no error, no gap in the logs, and a downstream table that is silently
missing a day.

In [ ]:
attempts = 0
delivered = False

while attempts < MAX_ATTEMPTS:
    result = flaky_log[attempts]
    attempts = attempts + 1
    print("attempt", attempts, "->", result)
    if result == "ok":
        delivered = True
        break

print("delivered:", delivered, "after", attempts, "attempts")

# What it must NOT do is carry on as if the delivery happened. A retry loop
# that runs out of attempts has FAILED, and the `delivered` flag is the only
# thing that says so -- the loop ending looks identical either way.

PART C — Validating row by row

### Question 5

Validation. -> `5 clean, 3 rejected, from 8 rows`. Rejected: `103 -- blank amount`, `105 -- negative`, `107 -- not a number`.

Three `continue`s in a cascade, each with its own reason recorded before it
fires. The order matters: the blank check has to come before the digit
check, because `"".isdigit()` is `False` and would report a blank field as
"not a number".

`"12.50".replace(".", "", 1).replace("-", "", 1)` strips **one** decimal
point and **one** minus sign, leaving `"1250"`, which is all digits. The
`1` argument is what makes it a check rather than a mangling — without it,
`"1.2.3"` would pass.

The important half is `rejects`. A validation loop that only counts is
useless at 3am: **record the id and the reason**, every time, or someone
has to re-run the whole feed to find out what went wrong.

And note row 104 survived. Its amount is fine; its *user* is not, and this
loop was never asked about that.

In [ ]:
clean = []
rejects = []

for row in feed:
    amount = row["amount"]

    if not amount:
        rejects.append((row["id"], "blank amount"))
        continue

    # strip one decimal point and one minus sign, then ask if what is left
    # is all digits
    if not amount.replace(".", "", 1).replace("-", "", 1).isdigit():
        rejects.append((row["id"], "not a number"))
        continue

    value = float(amount)
    if value < 0:
        rejects.append((row["id"], "negative"))
        continue

    clean.append({"id": row["id"], "ts": row["ts"],
                  "user": row["user"], "amount": value})

print(len(clean), "clean,", len(rejects), "rejected, from", len(feed), "rows")
for rid, reason in rejects:
    print("  rejected", rid, "--", reason)

### Question 6

The join. -> four rows joined; `4 joined, 1 orphaned: [104]`; **`amount lost to orphans: 31.0`**.

One row out of eight, and it took **31.00** with it — against 47.74 that
made it through. Nearly two fifths of the money, lost to a single user id
that was not on the customer table.

That is why the last line prints an amount and not just a count. `1
orphan` sounds like a rounding error. `£31 of £78.74` does not, and they
are the same fact.

The orphan is real-world normal: a customer created after the dimension
table was last refreshed, a test account, a typo. What is not normal is
dropping it silently. Every join you write should be able to answer *how
many rows did not match, and what were they worth*.

In [ ]:
joined = []
orphans = []
orphan_amount = 0.0

for row in clean:
    if row["user"] not in customers:
        orphans.append(row["id"])
        orphan_amount = orphan_amount + row["amount"]
        continue
    joined.append((row["id"], row["user"], customers[row["user"]], row["amount"]))

for entry in joined:
    print(entry)

print(len(joined), "joined,", len(orphans), "orphaned:", orphans)
print("amount lost to orphans:", orphan_amount)

PART D — Only doing the new work

### Question 7

The incremental run. -> `processed 5 of 8 rows`, `watermark moves from 15 to 44`.

Three rows had `ts` of 5, 9 and 14, all at or below the watermark, so they
were skipped. Five were newer and got processed, and the watermark advanced
to the highest `ts` seen.

This is how you avoid reprocessing history every night. It rests entirely
on the watermark being stored somewhere that survives the run — a file, a
table, a metadata store — and on it being updated **only** after the work
succeeded. Move it first and a mid-run failure loses those rows forever.

Note it tracks the highest `ts`, not the last row's `ts`. Those are the
same here because the feed is sorted; on an unsorted feed only one of them
is right.

In [ ]:
watermark = 15
processed = 0
high_ts = watermark

for row in feed:
    if row["ts"] <= watermark:
        continue
    processed = processed + 1
    if row["ts"] > high_ts:
        high_ts = row["ts"]

print("processed", processed, "of", len(feed), "rows")
print("watermark moves from", watermark, "to", high_ts)
watermark = high_ts

### Question 8

The same loop, run again. -> `processed 0 of 8 rows`, `watermark stays at 44`.

Correct. Nothing new has arrived, so nothing is processed, and running it
again is harmless — that property is called idempotence and it is what lets
you safely re-run a failed job.

And it is **indistinguishable from disaster**. "0 rows processed, no
errors" is what a correct second run prints, and it is also what prints
when the source file never arrived, when the upstream job silently failed,
when a filter was typed wrong, and when a date format changed so nothing
matched.

This is worksheet 05 Q11 and worksheet 04 Q9 wearing a suit: **a loop that
runs zero times looks exactly like a loop that had nothing to do.** The
only cure is to check the input separately — how many rows did the source
have? — and alert when that number is zero, not when the loop is.

In [ ]:
processed = 0
high_ts = watermark

for row in feed:
    if row["ts"] <= watermark:
        continue
    processed = processed + 1
    if row["ts"] > high_ts:
        high_ts = row["ts"]

print("processed", processed, "of", len(feed), "rows")
print("watermark stays at", watermark)

# There is no difference. "0 rows processed, no errors" is exactly what a
# correct second run prints AND exactly what an empty source file prints.
# The loop cannot tell you which happened -- worksheet 05 Q11 again. Only a
# separate check ("the source had N rows") can.

PART E — What the loop costs

### Question 9

Two ways to join. -> `nested loop: 15 comparisons`, `dict lookup: 5 lookups`, `same answer: True`.

Five rows against three customers is 5 × 3 = 15 comparisons for the nested
loop, and 5 for the dictionary — one per row, regardless of how many
customers there are.

That second clause is everything. Grow the customer table to 10,000 and
the nested loop does 50,000 comparisons while the dictionary still does 5.
This is worksheet 02 Q12's multiplication, and it is the difference between
a job that takes a second and one that takes an hour.

A dictionary lookup is roughly constant-time because of hashing — which is
also why keys must be hashable, and why worksheet 03 Q11 raised on a list.
The rule of thumb: **if you are looping over one collection inside a loop
over another, build a dictionary instead.**

In [ ]:
nested_result = []
comparisons = 0
for row in clean:
    for cust, region in customers.items():
        comparisons = comparisons + 1
        if row["user"] == cust:
            nested_result.append((row["id"], region))

lookup_result = []
lookups = 0
for row in clean:
    lookups = lookups + 1
    if row["user"] in customers:
        lookup_result.append((row["id"], customers[row["user"]]))

print("nested loop:", comparisons, "comparisons")
print("dict lookup:", lookups, "lookups")
print("same answer:", nested_result == lookup_result)

### Question 10

Progress every three records. -> `... 0 records processed`, `... 3`, `... 6`, then `done -- 8 records`.

`0 % 3` is `0`, so the very first pass reports — before a single record has
been handled. The line is false, and on a long job it is the line that
makes you think work started earlier than it did.

The other end is worse: records 7 and 8 produce no line at all, so a job
that died at record 7 would have `... 6 records processed` as its last word
and nothing to say it stopped. Progress logs go stale silently.

`enumerate(feed, start=1)` fixes the first record. The final `done` line —
outside the loop, always printed — is what covers the last few, and it is
the line people leave out.

In [ ]:
for i, row in enumerate(feed):
    if i % 3 == 0:
        print("...", i, "records processed")

print("done --", len(feed), "records")

# It reported at i = 0, 3 and 6: BEFORE doing anything, then after 3 and
# after 6. The first line is a lie -- nothing had been processed yet -- and
# the last three records got no line at all. enumerate(feed, start=1) fixes
# the first problem; the final "done" line is what covers the second.

PART F — The run report

### Question 11

Two true summaries. -> A: `rows delivered: 4`, `errors: 0`, `status: OK`. B: `8` in feed, `3` rejected, `1` orphaned, `4` delivered, `47.74` delivered, **`31.0` lost**, `accounted for: 8 of 8`.

Summary A is accurate in every particular. Four rows were delivered. Zero
errors were raised — genuinely zero, because every row that went missing
went missing through a `continue`, which is not an error. `status: OK`.

Half the feed did not arrive, and 31.00 of 78.74 — **39% of the money** —
is somewhere else.

Summary B differs in one structural way: it starts from `len(feed)` and
accounts for every row. `3 + 1 + 4 = 8`, and that line is the check. If
rejected plus orphaned plus delivered does not equal the number of rows
that came in, rows are being lost somewhere you have not looked yet.

So: **report the denominator, and reconcile to it.** A pipeline summary
that only counts successes cannot distinguish a good run from a broken one,
and it will pass unnoticed for months — which is roughly how long it takes
for someone downstream to ask why the north region looks light.

That is the whole session in one output. Nothing here raised, nothing was
mis-typed, every loop did exactly what it was told. Getting the code to run
is the easy half.

In [ ]:
delivered_rows = len(joined)
errors = 0

print("=== summary A ===")
print("rows delivered:", delivered_rows)
print("errors:", errors)
print("status: OK")

print()
print("=== summary B ===")
print("rows in feed:      ", len(feed))
print("rejected (bad data):", len(rejects))
print("orphaned (no user): ", len(orphans))
print("rows delivered:    ", delivered_rows)
print("amount delivered:  ", sum(r[3] for r in joined))
print("amount lost:       ", orphan_amount)
accounted = len(rejects) + len(orphans) + delivered_rows
print("accounted for:     ", accounted, "of", len(feed))
print("status: OK -- but", len(feed) - delivered_rows, "rows did not arrive")